# Test 06: EAI (External Access Integration) Setup

Tests the network rule generation and application flow.

**What this tests:**
1. SQL generation for different language combos
2. Host list correctness (shared + language-specific)
3. Applying EAI via Snowpark session (with privileges)
4. Graceful fallback when privileges are insufficient
5. SQL file export for admin handoff
6. Dynamic network rule application (no restart needed)

**Note:** Tests 3-4 require specific role configurations. Test 1-2 work
without any special privileges.

## 1. Install the Toolkit

In [ ]:
# Install from local source (notebooks/ is inside the package dir)
!pip install -q ..

## 2. Generate SQL -- R Only

Verify the generated SQL contains only shared + R-specific hosts.

In [ ]:
from sfnb_multilang import generate_eai_sql

sql_r = generate_eai_sql(languages=["r"], r_adbc=True)
print(sql_r)

In [ ]:
# Validate R-only SQL content
assert "micro.mamba.pm" in sql_r, "FAIL: shared host missing"
assert "conda.anaconda.org" in sql_r, "FAIL: conda-forge host missing"
assert "community.r-multiverse.org" in sql_r, "FAIL: ADBC host missing (r_adbc=True)"
assert "proxy.golang.org" in sql_r, "FAIL: Go proxy missing (needed for ADBC)"

# Should NOT contain Scala/Julia hosts
assert "repo1.maven.org" not in sql_r, "FAIL: Maven host should not be in R-only SQL"

print("PASS: R-only SQL has correct hosts")

## 3. Generate SQL -- All Languages

Verify deduplication: github.com should appear once even though
both Scala and Julia need it.

In [ ]:
sql_all = generate_eai_sql(languages=["r", "scala", "julia"])
print(sql_all)

In [ ]:
# Verify all-language SQL
assert "repo1.maven.org" in sql_all, "FAIL: Maven Central missing"
assert "github.com" in sql_all, "FAIL: GitHub missing"
assert "pypi.org" in sql_all, "FAIL: PyPI missing"

# Deduplication: github.com in VALUE_LIST only once
value_section = sql_all.split("VALUE_LIST")[1].split(";")[0]
assert value_section.count("github.com") == 1, "FAIL: github.com duplicated"

# Julia package server bypass: pkg.julialang.org should NOT be present
assert "pkg.julialang.org" not in sql_all, "FAIL: Julia pkg server should be bypassed"

print("PASS: All-language SQL has correct hosts with deduplication")

## 4. Generate SQL with Account

When an account is provided, the Snowflake endpoint should be included.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
account = session.sql("SELECT CURRENT_ACCOUNT()").collect()[0][0]
print(f"Account: {account}")

sql_with_acct = generate_eai_sql(languages=["r", "scala"], account=account)

assert f"{account}.snowflakecomputing.com" in sql_with_acct.lower() or \
       f"{account.lower()}.snowflakecomputing.com" in sql_with_acct.lower(), \
       "FAIL: Account endpoint not in SQL"

print(f"PASS: SQL includes {account}.snowflakecomputing.com")

## 5. Apply EAI via Session (Privileged Path)

This cell attempts to create the network rule and EAI directly.
It requires `CREATE INTEGRATION` and `CREATE NETWORK RULE` privileges.

If it fails due to insufficient privileges, the toolkit will:
1. Print the SQL to the output
2. Save it to `eai_setup.sql`
3. Print instructions for the admin

In [ ]:
from sfnb_multilang import apply_eai

sql = apply_eai(
    session,
    languages=["r", "scala"],
    account=account,
)

print("\nGenerated SQL length:", len(sql), "chars")

## 6. Verify EAI Was Created (if privileged)

Check if the integration exists. This will fail gracefully if the
previous step fell back to file export.

In [ ]:
try:
    result = session.sql("SHOW EXTERNAL ACCESS INTEGRATIONS LIKE 'multilang_notebook_eai'").collect()
    if result:
        print(f"EAI found: {result[0][0]}")
        print("PASS: EAI created successfully")
    else:
        print("INFO: EAI not found -- apply_eai likely fell back to file export")
except Exception as e:
    print(f"INFO: Could not check EAI: {e}")
    print("This is expected if your role cannot list integrations")

## 7. Verify SQL File Export (Fallback Path)

If apply_eai failed due to permissions, it should have created
`eai_setup.sql` in the working directory.

In [ ]:
import os

sql_file = "./eai_setup.sql"
if os.path.isfile(sql_file):
    with open(sql_file) as f:
        content = f.read()
    print(f"SQL file exists ({len(content)} bytes)")
    print(f"Contains NETWORK RULE: {'CREATE OR REPLACE NETWORK RULE' in content}")
    print(f"Contains INTEGRATION: {'CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION' in content}")
    print("\nPASS: SQL file exported for admin handoff")
else:
    print("INFO: No SQL file -- apply_eai likely succeeded directly")

## 8. Test Dynamic Application

If the EAI was just created, verify outbound network access works
immediately (no kernel restart).

**Important:** You must first enable the integration in Snowsight:
Notebook settings > External access > toggle on `multilang_notebook_eai`

In [ ]:
import urllib.request

test_urls = [
    ("conda-forge", "https://conda.anaconda.org/conda-forge"),
    ("PyPI", "https://pypi.org/simple/"),
]

for name, url in test_urls:
    try:
        req = urllib.request.Request(url, method="HEAD")
        urllib.request.urlopen(req, timeout=5)
        print(f"  {name:15s} OK")
    except Exception as e:
        print(f"  {name:15s} BLOCKED ({type(e).__name__})")

print("\nIf any hosts show BLOCKED, enable the EAI in Notebook settings")

## 9. CLI-Style EAI Generation

Test the CLI interface for generating EAI SQL.

In [ ]:
!python -m sfnb_multilang generate-eai --r --scala --account {account}

## Test Summary

| Test | Expected |
|---|---|
| R-only SQL | Shared + R hosts, no Maven |
| All-language SQL | All hosts, github.com deduplicated, no pkg.julialang.org |
| Account in SQL | `<account>.snowflakecomputing.com` present |
| apply_eai | Creates EAI or falls back to .sql file |
| SQL file export | Contains valid CREATE statements |
| Dynamic access | Network reachable without restart |
| CLI | `generate-eai` subcommand works |